In [1]:
import os
from pathlib import Path

def has_project_files(path: Path) -> bool:
    try:
        return (path / "config" / "config.yaml").is_file() and (path / "src").is_dir()
    except OSError:
        return False

def find_project_root(start: Path = Path.cwd()) -> Path:
    try:
        start = start.resolve()
    except OSError:
        start = Path.cwd()

    seeds = [start, Path.home() / "Documents" / "projects" / "Summarizer"]
    candidates = []
    for seed in seeds:
        try:
            seed = seed.resolve()
        except OSError:
            pass
        for candidate in [seed, *seed.parents]:
            if candidate not in candidates:
                candidates.append(candidate)
    for candidate in candidates:
        if has_project_files(candidate):
            return candidate

    for candidate in candidates:
        try:
            children = list(candidate.iterdir())
        except OSError:
            continue
        for child in children:
            try:
                if child.is_dir() and has_project_files(child):
                    return child
            except OSError:
                continue
    raise FileNotFoundError("Could not find project root containing config/config.yaml")

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
PROJECT_ROOT

PosixPath('/Users/tyrion/Documents/projects/Summarizer')

In [2]:
Path.cwd()

PosixPath('/Users/tyrion/Documents/projects/Summarizer')

In [3]:
PROJECT_ROOT

PosixPath('/Users/tyrion/Documents/projects/Summarizer')

In [4]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class ModelTrainerConfig:
    root_dir: Path
    data_path: Path
    model_ckpt: str
    num_train_epochs: int
    warmup_steps: int
    per_device_train_batch_size: int
    per_device_eval_batch_size: int
    weight_decay: float
    logging_steps: int
    eval_strategy: str
    eval_steps: int
    save_steps: int
    gradient_accumulation_steps: int



In [5]:
from src.Text_Summarizer.constants import *
from src.Text_Summarizer.utils.common import read_yaml, create_directories




In [6]:
class ConfigurationManager:
    def __init__(self, config_file=CONFIG_FILE_PATH, params_file=PARAMS_FILE_PATH):
        self.config_file = config_file
        self.params_file = params_file
        self.config = read_yaml(config_file)
        self.params = read_yaml(params_file)

    def get_model_trainer_config(self)->ModelTrainerConfig:
        config = self.config.model_trainer
        params = self.params.TrainingArguments

        root_dir = Path(config.root_dir)
        data_path = Path(config.data_path)
        if not root_dir.is_absolute():
            root_dir = PROJECT_ROOT / root_dir
        if not data_path.is_absolute():
            data_path = PROJECT_ROOT / data_path

        create_directories([root_dir])
        model_trainer_config = ModelTrainerConfig(
            root_dir=root_dir,
            data_path=data_path,
            model_ckpt=config.model_ckpt,
            num_train_epochs=params.num_train_epochs,
            warmup_steps=params.warmup_steps,
            per_device_train_batch_size=params.per_device_train_batch_size,
            per_device_eval_batch_size=params.per_device_eval_batch_size,
            weight_decay=params.weight_decay,
            logging_steps=params.logging_steps,
            eval_strategy=params.eval_strategy,
            eval_steps=params.eval_steps,
            save_steps=params.save_steps,
            gradient_accumulation_steps=params.gradient_accumulation_steps,
        )
        return model_trainer_config

    



In [7]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorForSeq2Seq
import torch
from datasets import load_from_disk

/Users/tyrion/Documents/projects/Summarizer/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config

    def train(self):
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        tokenizer = AutoTokenizer.from_pretrained(self.config.model_ckpt)
        model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(self.config.model_ckpt).to(device)
        seq2seq_data_collator = DataCollatorForSeq2Seq(tokenizer, model=model_pegasus)

        dataset_samsum_pt = load_from_disk(self.config.data_path)

        trainer_args = TrainingArguments(
            output_dir=str(self.config.root_dir),
            num_train_epochs=self.config.num_train_epochs,
            warmup_steps=self.config.warmup_steps,
            per_device_train_batch_size=self.config.per_device_train_batch_size,
            per_device_eval_batch_size=self.config.per_device_eval_batch_size,
            weight_decay=self.config.weight_decay,
            logging_steps=self.config.logging_steps,
            eval_strategy=self.config.eval_strategy,
            eval_steps=self.config.eval_steps,
            save_steps=self.config.save_steps,
            gradient_accumulation_steps=self.config.gradient_accumulation_steps,
        )

        trainer = Trainer(
            model=model_pegasus,
            args=trainer_args,
            processing_class=tokenizer,
            data_collator=seq2seq_data_collator,
            train_dataset=dataset_samsum_pt["train"],
            eval_dataset=dataset_samsum_pt["validation"],
        )

        trainer.train()

        model_pegasus.save_pretrained(os.path.join(str(self.config.root_dir), "pegasus-samsum-model"))
        tokenizer.save_pretrained(os.path.join(str(self.config.root_dir), "tokenizer"))

In [ ]:
!pip install --upgrade accelerate
!pip uninstall -y transformers accelerate
!pip install transformers accelerate
!pip install transformers[sentencepiece] datasets sacrebleu rouge_score py7zr -q

In [10]:
model_dir = PROJECT_ROOT / "artifacts" / "model_trainer" / "pegasus-samsum-model"
if not model_dir.exists():
    raise FileNotFoundError(f"Saved model folder not found: {model_dir}")

model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(str(model_dir))

Loading weights: 100%|██████████| 680/680 [00:00<00:00, 11625.34it/s]
